# Step 10 — Convert legacy DMX waveforms and review partial recoveries

Run this notebook **or** the command-line conversion, rather than both for the same run. Use a kernel with the updated FLOVOpy and ObsPy, and restart it after updating FLOVOpy.

## Workflow and safeguards

1. Load `conversion_config.json` and inspect input/output locations and options.
2. Convert the raw event files; review the saved index, station-ID provenance and failures.
3. Optionally save separate diagnostic recovery candidates and/or inspect recovered traces. Both optional actions are disabled by default.
4. Run step 11 for waveform availability plots, then step 15 for network history and StationXML.

Raw legacy files are read without modification. Production MiniSEED waveforms are written into the configured SEISAN WAV directory, with metadata under the same external `output_root`. SEISAN describes the database layout here; the waveform encoding is MiniSEED.

The converter removes IRIG traces, applies legacy station-ID mapping and subtracts the 2048-count offset. Setting the sample rate to 100 Hz corrects header metadata; it does not resample the samples or establish absolute timing. Recorded clocks, offsets and drift still need separate evaluation. Mapping preserves `PI2L → XB.PI2..ELZ` versus `PI2Z → XB.PI2..EHZ`; this mapping is not evidence that PI2L was recorded in DMX.

## 1. Load configuration

The following cell imports the converter and resolves paths; it does not run conversion. `CONFIG_FILE=None` selects the normal configuration. Inputs and outputs come from the shared path helper, not notebook-specific paths.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Start in 10_conversion or the project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths, load_conversion_config
from flovopy.core.vdap import convert_dmx_to_seisan
CONFIG_FILE = None
P = pipeline_paths(config_file=CONFIG_FILE)
CFG = load_conversion_config(P['CONFIG_FILE'])
OPTIONS = CFG.get('waveform', {})
DATA_TOP = Path(P['DATA_TOP'])
SUDS_TOP = Path(P['SUDS_TOP'])
FAIR_TOP = Path(P['FAIR_TOP'])
NETWORK, DATABASE = P['NET'], P['DB']
SEISAN_WAV_DB = Path(P['SEISAN_WAV_DB'])
FAIR_META_DIR, TMP_DIR = Path(P['FAIR_META_DIR']), Path(P['TMP_DIR'])
SAMPLING_RATE = OPTIONS.get('sampling_rate', 100.0)
PATTERN = OPTIONS.get('pattern', '**/*.DMX')
MAX_FILES = OPTIONS.get('max_files')
OVERWRITE = OPTIONS.get('overwrite', False)
VERBOSE = OPTIONS.get('verbose', False)
RECOVER_PARTIAL = OPTIONS.get('recover_partial', False)
if not SUDS_TOP.is_dir():
    raise FileNotFoundError(f'Check data_top/legacy_root in {P["CONFIG_FILE"]}: {SUDS_TOP}')
print('Configuration:', P['CONFIG_FILE'])
print('Inputs:', SUDS_TOP)
print('External outputs:', P['OUTPUT_ROOT'])


## 2. Review the run settings

`overwrite=False` normally reuses compatible existing outputs. Changing the recovery option does not itself regenerate existing files. Avoid enabling overwrite globally unless you intend to regenerate the archive. `max_files` limits a trial run; reset it before a full run.

`waveform.recover_partial` controls production fallback (default false if omitted). Normal reading is attempted first. If it fails and fallback is enabled, FLOVOpy retains only a conservatively validated prefix; it does not scan past corruption to invent a later continuation. Without independent ASCII comparison, the trace immediately before the damaged boundary is also discarded. A recovered prefix is incomplete and is not automatically a scientifically usable event.

In [ ]:
if not SUDS_TOP.is_dir():
    raise FileNotFoundError(f"Check DATA_TOP: DMX input folder not found: {SUDS_TOP}")

print(f"Input: {SUDS_TOP}")
print(f"Waveforms: {SEISAN_WAV_DB}")
print(f"Index: {TMP_DIR / '10_waveform_index.csv'}")
print(f"Overwrite existing waveforms: {OVERWRITE}")


In [ ]:
print('Sampling rate override:', SAMPLING_RATE)
print('Pattern / file limit:', PATTERN, MAX_FILES)
print('Partial recovery enabled:', RECOVER_PARTIAL)
print('Metadata:', FAIR_META_DIR)
print('Failure list:', TMP_DIR / '10_conversion_failures.csv')

## 3. Run production conversion

This cell writes waveforms and replaces the run's index, mapping/provenance tables and failure list. Preserve an earlier failure CSV if you want to compare runs: successfully recovered files may disappear from the new failure list. Recovery metadata, when present, is recorded in `partial_recovery` and `recovery_details` columns of the waveform index. Review warnings and the summary below.

In [ ]:
result = convert_dmx_to_seisan(
    rawtop=SUDS_TOP,
    seisan_wav_db=SEISAN_WAV_DB,
    network=NETWORK,
    database=DATABASE,
    sampling_rate=SAMPLING_RATE,
    pattern=PATTERN,
    max_files=MAX_FILES,
    overwrite=OVERWRITE,
    remove_irig=True,
    recover_partial=RECOVER_PARTIAL,
    continue_on_error=True,
    verbose=VERBOSE,
    out_waveform_index=TMP_DIR / "10_waveform_index.csv",
    out_trace_id_map=FAIR_META_DIR / "10_trace_id_mapping.csv",
    out_trace_id_map_tex=FAIR_META_DIR / "10_trace_id_mapping.tex",
    out_trace_id_provenance=FAIR_META_DIR / "10_trace_id_provenance.csv",
    out_failures=TMP_DIR / "10_conversion_failures.csv",
)


## 4. Review conversion results

The index is read back from saved MiniSEED headers, so counts reflect the merged output. Status distinguishes writes and reuse. Failures report source file and processing stage; filename collisions are errors. The mapping table contains verified saved IDs, while provenance also records attempted mappings for failed conversions.

Existing files with incompatible IDs or sample rates are rejected. Investigate before selectively regenerating them. For other archives the converter accepts `event_id_func(path)` and `id_fixer(trace, network=...)`; this notebook uses the Pinatubo defaults.

In [ ]:
display(result["summary"])
display(result["waveform_index"].head())
display(result["trace_id_map"])
display(result["failures"])


## 5. Optional diagnostic settings — disabled by default

These settings do not alter production conversion. Set either switch to `True` only when needed. You can run the setup and optional cells against an existing conversion without rerunning section 3.

`DIAGNOSTIC_FAILURES` may point to a preserved earlier CSV. The inspection also includes sources flagged as partial recoveries in the current waveform index, since they may no longer appear among failures. `EXTRA_DMX_FILES` adds explicit sources. An optional `ASCII_DIR` contains independent DOS SUD2ASC `.ASC` exports; these are comparisons, not assumed ground truth.

Each enabled action writes to a new timestamped directory beneath `output_root/recovery_diagnostics`. Candidate files are never inserted into the production archive.

In [ ]:
RUN_SAVE_CANDIDATES = False
RUN_INSPECT_RECOVERIES = False
DIAGNOSTIC_FAILURES = TMP_DIR / '10_conversion_failures.csv'
ASCII_DIR = None  # Path to independent SUD2ASC exports, if available.
EXTRA_DMX_FILES = []  # Optional raw DMX paths for inspection.
MAX_INSPECTION_FILES = None  # Set a small integer for a trial; None inspects all.
MAX_INLINE_PLOTS = 4  # All plots are saved; only this many are displayed.

## 6. Optional: save separate prefix candidates (formerly step 12)

This calls the existing `pinatubo_steps.dmx_recovery` implementation. It saves candidate prefix DMX files, compressed sample arrays and a manifest with source hashes, structural checks and any ASCII comparisons. These diagnostic candidates retain original samples and IDs: no count-offset subtraction, resampling, merging or clock correction.

Candidate extraction is deliberately distinct from the production reader's conservative boundary guard. A candidate is material for inspection, not approval for publication or automatic inclusion in the archive. Errors are recorded per file so remaining files can still be examined.

In [ ]:
if not RUN_SAVE_CANDIDATES:
    print('Candidate extraction disabled; set RUN_SAVE_CANDIDATES=True to run.')
else:
    from datetime import datetime, timezone
    from pinatubo_steps.dmx_recovery import run as save_recovery_candidates
    if not Path(DIAGNOSTIC_FAILURES).is_file():
        raise FileNotFoundError(DIAGNOSTIC_FAILURES)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    candidate_out = Path(P['OUTPUT_ROOT']) / 'recovery_diagnostics' / 'dmx' / stamp
    candidate_results = save_recovery_candidates(
        Path(DIAGNOSTIC_FAILURES), candidate_out,
        ascii_dir=Path(ASCII_DIR) if ASCII_DIR else None,
    )
    print('Candidate files:', sum(r['status'].startswith('candidate') for r in candidate_results))
    print('Manifest:', candidate_out / 'recovery_manifest.json')
    display(candidate_results)

## 7. Optional: plot raw recovered traces (formerly step 13)

This uses FLOVOpy `read_dmx_file(fix=False, recover_partial=True)`: normal reading first, then the conservative prefix fallback, with optional ASCII checks. These are raw sample plots, without production ID/count corrections. Each file is handled independently; unreadable files and plotting errors are retained in the inspection manifest. Flat channels are counted, not silently removed. No production waveforms are replaced.

Inspect coherent signals, flat/dead channels, spikes, clipping, station labels and timing uncertainty. A recovered prefix cannot restore missing channels or the lost remainder. Regular reading on a previously failed source may now succeed; its manifest entry will say it is not partial.

In [ ]:
if not RUN_INSPECT_RECOVERIES:
    print('Inspection disabled; set RUN_INSPECT_RECOVERIES=True to run.')
else:
    import csv, json
    from datetime import datetime, timezone
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import Image
    from flovopy.core.vdap import read_dmx_file

    sources = [str(path) for path in EXTRA_DMX_FILES]
    failures_path = Path(DIAGNOSTIC_FAILURES)
    if failures_path.is_file():
        with failures_path.open(newline='') as handle:
            sources.extend(row['dmx_file'] for row in csv.DictReader(handle))
    index_path = TMP_DIR / '10_waveform_index.csv'
    if index_path.is_file():
        index = pd.read_csv(index_path, low_memory=False)
        if {'partial_recovery', 'dmx_file'}.issubset(index.columns):
            partial = index['partial_recovery'].astype(str).str.lower().isin(['true', '1', '1.0'])
            sources.extend(index.loc[partial, 'dmx_file'].dropna().astype(str))
    sources = list(dict.fromkeys(sources))
    if MAX_INSPECTION_FILES is not None:
        sources = sources[:MAX_INSPECTION_FILES]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    inspection_out = Path(P['OUTPUT_ROOT']) / 'recovery_diagnostics' / 'obspy_inspection' / stamp
    inspection_out.mkdir(parents=True, exist_ok=False)
    inspection_results = []
    shown = 0
    for name in sources:
        source = Path(name)
        entry = {'file': str(source)}
        fig = None
        try:
            ascii_file = Path(ASCII_DIR) / (source.stem + '.ASC') if ASCII_DIR else None
            stream = read_dmx_file(source, fix=False, recover_partial=True, recovery_ascii=ascii_file)
            entry.update(traces=len(stream), partial=any(hasattr(tr.stats, 'dmx_recovery') for tr in stream))
            entry['recovery'] = [dict(tr.stats.dmx_recovery) for tr in stream if hasattr(tr.stats, 'dmx_recovery')]
            entry['status'] = 'readable' if stream else 'no_usable_traces'
            entry['constant_traces'] = sum(bool(tr.data.size and (tr.data == tr.data[0]).all()) for tr in stream)
            if stream:
                fig = stream.plot(show=False, handle=True, automerge=False,
                                  equal_scale=False, method='full',
                                  size=(1200, max(350, 150 * len(stream))))
                fig.suptitle(source.name + ' — raw samples; recorded time not independently verified')
                image = inspection_out / f'{len(inspection_results):04d}_{source.stem}.png'
                fig.savefig(image)
                entry['plot'] = str(image)
                if shown < MAX_INLINE_PLOTS:
                    display(Image(filename=str(image)))
                    shown += 1
        except Exception as exc:
            entry.update(status='error', error=f'{type(exc).__name__}: {exc}')
        finally:
            if fig is not None:
                plt.close(fig)
        inspection_results.append(entry)
        (inspection_out / 'inspection_manifest.json').write_text(
            json.dumps(inspection_results, indent=2, default=str) + '\n')
    if not sources:
        (inspection_out / 'inspection_manifest.json').write_text('[]\n')
        print('No failed/flagged sources found. Supply an older failure CSV or EXTRA_DMX_FILES if needed.')
    print('Inspection output:', inspection_out)
    display(pd.DataFrame(inspection_results))

## 8. Continue the pipeline

Review the production summary, remaining failures and partial-recovery flags before continuing. Run **11** for waveform counts and station availability, then **15** for archive-informed network metadata and StationXML. Keep partial recovery explicitly identified in downstream analyses; these records do not establish catalogue completeness or clock accuracy.
